In [4]:
import piplite
await piplite.install('seaborn')

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('talabat_enhanced_orders2.csv')

In [6]:
drop_cols = ["Delivery_Time", "Delivery_Duration_Minutes", "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID"]
df_clean = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [7]:
def apply_feature_engineering(data, top_k=5):
    df_fe = data.copy()
    
    df_fe["Order_Time"] = pd.to_datetime(df_fe["Order_Time"])
    df_fe["order_hour"] = df_fe["Order_Time"].dt.hour
    df_fe["order_dayofweek"] = df_fe["Order_Time"].dt.dayofweek
    df_fe["is_weekend"] = df_fe["order_dayofweek"].isin([5,6]).astype(int)
    df_fe["is_peak_hour"] = df_fe["order_hour"].isin(list(range(12,16)) + list(range(19,24))).astype(int)
    
    df_fe["price_per_item"] = df_fe["Total_Price"] / df_fe["Quantity"]
    df_fe["price_tier"] = pd.cut(df_fe["Total_Price"], bins=[0, 100, 250, 500, np.inf], labels=["low","medium","high","very_high"])
    
    top_items = df_fe["Item_Name"].value_counts().head(top_k).index
    df_fe["Item_Name_reduced"] = np.where(df_fe["Item_Name"].isin(top_items), df_fe["Item_Name"], "Other")
    
    df_fe = df_fe.drop(columns=["Order_Time", "Item_Name"])
    return df_fe

df_final = apply_feature_engineering(df_clean, top_k=5)

In [8]:
X = df_final.drop(columns=["Order_Status"])
y = df_final["Order_Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Accuracy: 0.4034


In [9]:
ohe_names = model.named_steps["preprocess"].named_transformers_["cat"].get_feature_names_out(cat_cols)
all_names = np.concatenate([ohe_names, num_cols])

fi = pd.DataFrame({
    "feature": all_names,
    "importance": model.named_steps["classifier"].feature_importances_
}).sort_values("importance", ascending=False)

print(fi.head(10))

                 feature  importance
36  Delivery_Distance_km    0.077508
34            Driver_Lat    0.075110
33          Customer_Lon    0.074974
35            Driver_Lon    0.074713
41        price_per_item    0.074584
29           Total_Price    0.074547
30        Restaurant_Lat    0.074452
32          Customer_Lat    0.074221
31        Restaurant_Lon    0.074050
37            order_hour    0.053025


In [10]:
k_values = [3, 5, 6]
results = []

for k in k_values:
    temp_df = apply_feature_engineering(df_clean, top_k=k)
    
    X_k = temp_df.drop(columns=["Order_Status"])
    y_k = temp_df["Order_Status"]
    X_tr, X_te, y_tr, y_te = train_test_split(X_k, y_k, test_size=0.2, random_state=42, stratify=y_k)
    
    model.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te))
    
    results.append({"k": k, "accuracy": acc})
    print(f"k={k} completed. Accuracy: {acc:.4f}")

comparison_df = pd.DataFrame(results)
print(comparison_df)

k=3 completed. Accuracy: 0.3908
k=5 completed. Accuracy: 0.4034
k=6 completed. Accuracy: 0.3950
   k  accuracy
0  3    0.3908
1  5    0.4034
2  6    0.3950
